In [2]:
import tkinter as tk
from tkinter import ttk, messagebox
import pickle
import os
from datetime import datetime

# ==========================================
# 1. BACKEND - DATA STORAGE & CALCULATION LOGIC
# ==========================================
class CafeDataManager:
    def __init__(self, filename="ultra_cafe_data.pkl"):
        self.filename = filename
        self.load_database()

    def load_database(self):
        """Loads data from the pickle file or initializes it if empty."""
        if os.path.exists(self.filename):
            try:
                with open(self.filename, 'rb') as f:
                    self.db = pickle.load(f)
            except Exception:
                self.init_new_db()
        else:
            self.init_new_db()

    def init_new_db(self):
        self.db = {
            "active_cabins": {},   # Currently sitting customers
            "pc_history": [],      # Past completed PC logs
            "other_sales": []      # Miscellaneous item logs
        }
        self.save_database()

    def save_database(self):
        """Saves current state to the pickle file."""
        with open(self.filename, 'wb') as f:
            pickle.dump(self.db, f)

    def calculate_rate(self, start_str, end_str):
        """Applies pricing: 40 Rs for <= 30 mins, 70 Rs/hr for > 30 mins."""
        fmt = "%Y-%m-%d %H:%M:%S"
        t1 = datetime.strptime(start_str, fmt)
        t2 = datetime.strptime(end_str, fmt)
        
        diff_mins = int((t2 - t1).total_seconds() / 60)
        if diff_mins <= 0:
            return 0, 0
        
        if diff_mins <= 30:
            charge = 40.0
        else:
            charge = round((diff_mins / 60.0) * 70.0, 2)
            
        return diff_mins, charge

# ==========================================
# 2. FRONTEND - USER INTERFACE WITH TABS
# ==========================================
class UltraGamingZoneApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Ultra Gaming Zone - Advanced DBMS Dashboard")
        self.root.geometry("1000x650")
        self.root.configure(bg="#1a1a1a")

        self.manager = CafeDataManager()

        # Application Header
        header = tk.Label(self.root, text="ULTRA GAMING ZONE", font=("Arial Black", 24, "bold"), fg="#00ffcc", bg="#1a1a1a")
        header.pack(fill="x", pady=10)

        # Tab Controller System
        self.notebook = ttk.Notebook(self.root)
        self.notebook.pack(fill="both", expand=True, padx=15, pady=15)

        # Creating the Screen Tabs
        self.tab_dashboard = tk.Frame(self.notebook, bg="#242424")
        self.tab_history = tk.Frame(self.notebook, bg="#242424")

        self.notebook.add(self.tab_dashboard, text=" Live Counter Dashboard ")
        self.notebook.add(self.tab_history, text=" Past Transaction History Ledger ")

        # Build interfaces
        self.build_dashboard_tab()
        self.build_history_tab()
        
        # Initial data push to UI
        self.refresh_live_table()
        self.update_totals_panel()

    def build_dashboard_tab(self):
        """Builds the main control desk screen."""
        control_frame = tk.LabelFrame(self.tab_dashboard, text=" Action Panel ", fg="white", bg="#242424", font=("Arial", 11, "bold"))
        control_frame.pack(side="left", fill="y", padx=15, pady=15)

        tk.Label(control_frame, text="Cabin / PC #:", bg="#242424", fg="white", font=("Arial", 10)).pack(pady=(15, 2), anchor="w", padx=10)
        self.entry_cabin = tk.Entry(control_frame, font=("Arial", 12), width=18)
        self.entry_cabin.pack(padx=10, pady=5)

        btn_start = tk.Button(control_frame, text="▶ START TIMING", bg="#28a745", fg="white", font=("Arial", 10, "bold"), command=self.start_timer, width=18, height=2)
        btn_start.pack(padx=10, pady=15)

        btn_end = tk.Button(control_frame, text="⏹ STOP & BILL PC", bg="#dc3545", fg="white", font=("Arial", 10, "bold"), command=self.end_timer, width=18, height=2)
        btn_end.pack(padx=10, pady=5)

        tk.Label(control_frame, text="───────────────────", bg="#242424", fg="#555555").pack(pady=20)

        tk.Label(control_frame, text="Other Sale Item Name:", bg="#242424", fg="white", font=("Arial", 10)).pack(anchor="w", padx=10)
        self.entry_misc_item = tk.Entry(control_frame, font=("Arial", 11), width=18)
        self.entry_misc_item.pack(padx=10, pady=5)

        tk.Label(control_frame, text="Item Cost (Rs):", bg="#242424", fg="white", font=("Arial", 10)).pack(anchor="w", padx=10)
        self.entry_misc_price = tk.Entry(control_frame, font=("Arial", 11), width=18)
        self.entry_misc_price.pack(padx=10, pady=5)

        btn_misc = tk.Button(control_frame, text="＋ ADD SALE", bg="#007bff", fg="white", font=("Arial", 10, "bold"), command=self.submit_misc_sale, width=18)
        btn_misc.pack(padx=10, pady=15)

        right_view = tk.Frame(self.tab_dashboard, bg="#242424")
        right_view.pack(side="right", fill="both", expand=True, padx=15, pady=15)

        tk.Label(right_view, text="Occupied PCs / Running Cabins:", font=("Arial", 12, "bold"), bg="#242424", fg="#00ffcc").pack(anchor="w", pady=(0, 5))
        
        self.live_tree = ttk.Treeview(right_view, columns=("Cabin", "Start"), show="headings", height=15)
        self.live_tree.heading("Cabin", text="Cabin # ID")
        self.live_tree.heading("Start", text="Login Timestamp / Time In")
        self.live_tree.column("Cabin", width=120, anchor="center")
        self.live_tree.column("Start", width=300, anchor="center")
        self.live_tree.pack(fill="both", expand=True)

        self.lbl_revenue = tk.Label(
            right_view, 
            text="Daily: Rs 0.0 | Monthly: Rs 0.0", 
            font=("Arial", 14, "bold"), 
            bg="#3a3a3a", 
            fg="#ffcc00", 
            padx=10, 
            pady=10
        )
        self.lbl_revenue.pack(fill="x", pady=(15, 0))

    def build_history_tab(self):
        """Builds the past transactions tab layout."""
        top_bar = tk.Frame(self.tab_history, bg="#242424")
        top_bar.pack(fill="x", padx=15, pady=10)

        tk.Label(top_bar, text="All Historic Log entries & Sales Book Transactions", font=("Arial", 13, "bold"), bg="#242424", fg="white").pack(side="left")
        
        btn_reload = tk.Button(top_bar, text="🔄 Reload History Data", bg="#6c757d", fg="white", command=self.populate_history_tables)
        btn_reload.pack(side="right", padx=5)

        hist_notebook = ttk.Notebook(self.tab_history)
        hist_notebook.pack(fill="both", expand=True, padx=15, pady=15)

        self.hist_pc_frame = tk.Frame(hist_notebook, bg="#242424")
        self.hist_sales_frame = tk.Frame(hist_notebook, bg="#242424")

        hist_notebook.add(self.hist_pc_frame, text=" Past Computer Use Sessions ")
        hist_notebook.add(self.hist_sales_frame, text=" Past Miscellaneous Sales Records ")

        self.history_pc_tree = ttk.Treeview(self.hist_pc_frame, columns=("Date", "Cabin", "Start", "End", "Mins", "Rs"), show="headings")
        self.history_pc_tree.heading("Date", text="Log Date")
        self.history_pc_tree.heading("Cabin", text="Cabin #")
        self.history_pc_tree.heading("Start", text="Time In")
        self.history_pc_tree.heading("End", text="Time Out")
        self.history_pc_tree.heading("Mins", text="Duration (Mins)")
        self.history_pc_tree.heading("Rs", text="Calculated Bill (Rs)")
        
        for col in ("Date", "Cabin", "Start", "End", "Mins", "Rs"):
            self.history_pc_tree.column(col, anchor="center")
        self.history_pc